# Meeting Transcription with Speaker Diarization

This notebook processes audio files to create timestamped transcriptions with speaker identification using pyannote-audio and speech_recognition.

## Setup Requirements

1. Install required packages:
```bash
pip install -r requirements.txt
```

2. For pyannote-audio, you need to:
   - Accept user conditions at: https://huggingface.co/pyannote/speaker-diarization-3.1
   - Get a HuggingFace token from: https://huggingface.co/settings/tokens
   - Copy `.env.sample` to `.env` and set the `USE_AUTH_TOKEN` variable with your actual token

3. Make sure you have ffmpeg installed for audio conversion

## Workflow

This notebook follows these steps:

1. **Audio Conversion**: Convert M4A to WAV format for processing
2. **Speaker Diarization**: Use pyannote-audio to identify who speaks when
3. **Speech Recognition**: Transcribe each speaker segment with `faster-whisper`
4. **Summary Generation**: Use Ollama to generate a meeting summary

The main advantage of this approach is that we get accurate speaker identification with timestamps, making the transcription much more useful for meeting analysis.

In [ ]:
# convert
from pydub import AudioSegment

file_path = "meetings/2026-04-08 15-33-29.mp4"
input_m4a_file = file_path
output_wav_file = file_path.replace(".mp4", ".wav")

audio = AudioSegment.from_file(input_m4a_file, format="mp4")
audio.export(output_wav_file, format="wav")

In [ ]:
# Optional: Preprocess audio for faster diarization
# Uncomment the following code to optimize audio for faster processing

def preprocess_audio_for_speed(input_file, output_file):
    """
    Preprocess audio to speed up diarization:
    - Convert to mono (50% speed improvement)
    - Downsample to 16kHz (another 30-50% improvement)
    - Normalize audio levels
    """
    print("Preprocessing audio for faster diarization...")
    
    audio = AudioSegment.from_file(input_file)
    
    # Convert to mono
    if audio.channels > 1:
        audio = audio.set_channels(1)
        print("Converted to mono")
    
    # Downsample to 16kHz (pyannote works well with this)
    if audio.frame_rate > 16000:
        audio = audio.set_frame_rate(16000)
        print(f"Downsampled to 16kHz (from {AudioSegment.from_file(input_file).frame_rate}Hz)")
    
    # Normalize audio
    audio = audio.normalize()
    print("Normalized audio levels")
    
    # Export optimized version
    audio.export(output_file, format="wav")
    print(f"Optimized audio saved as: {output_file}")
    
    return audio

# Uncomment these lines to use preprocessing:
optimized_wav_file = file_path + "_optimized.wav"
preprocess_audio_for_speed(output_wav_file, optimized_wav_file)
output_wav_file = optimized_wav_file  # Use optimized version for diarization

In [ ]:
import os
from pyannote.audio import Pipeline
import torch
import time
from pydub import AudioSegment
from dotenv import load_dotenv

load_dotenv()

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Get audio duration for time estimation
audio_info = AudioSegment.from_file(output_wav_file)
duration_minutes = len(audio_info) / (1000 * 60)
print(f"Audio duration: {duration_minutes:.1f} minutes")

# Estimate processing time
if device.type == "cuda":
    estimated_time = duration_minutes * 2  # ~2x real-time with GPU
else:
    estimated_time = duration_minutes * 8  # ~8x real-time with CPU
print(f"Estimated processing time: {estimated_time:.1f} minutes")

# Initialize the speaker diarization pipeline with device specification
print("Loading diarization model...")
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=os.getenv("USE_AUTH_TOKEN", None),
)

# Move pipeline to GPU if available
if device.type == "cuda":
    pipeline = pipeline.to(device)
    print("Pipeline moved to GPU")

# Perform speaker diarization with timing
print("Performing speaker diarization...")
start_time = time.time()

diarization = pipeline(output_wav_file)

diarization_time = time.time() - start_time
print(f"Diarization completed in {diarization_time:.1f} seconds ({diarization_time/60:.1f} minutes)")
print(f"Processing speed: {duration_minutes / (diarization_time/60):.1f}x real-time")

# Store speaker segments for later use
speaker_segments = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    speaker_segments.append({
        'start': turn.start,
        'end': turn.end,
        'speaker': speaker
    })
    print(f"start={turn.start:.1f}s stop={turn.end:.1f}s speaker_{speaker}")

print(f"Found {len(speaker_segments)} speaker segments")

# Free GPU memory
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Save speaker segments as backup (in case of crashes or interruptions)
import pickle
import json
from datetime import datetime

# Create backup filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_filename = f"speaker_segments_backup_{timestamp}.pkl"
json_filename = f"speaker_segments_backup_{timestamp}.json"

# Save as pickle (Python-specific, preserves exact data types)
try:
    with open(backup_filename, 'wb') as f:
        pickle.dump(speaker_segments, f)
    print(f"✓ Speaker segments saved as pickle: {backup_filename}")
except Exception as e:
    print(f"❌ Error saving pickle file: {e}")

# Also save as JSON (human-readable, cross-platform)
try:
    # Convert to JSON-serializable format
    json_data = {
        'metadata': {
            'created': datetime.now().isoformat(),
            'total_segments': len(speaker_segments),
            'audio_file': output_wav_file if 'output_wav_file' in globals() else 'unknown'
        },
        'segments': speaker_segments
    }
    
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)
    print(f"✓ Speaker segments saved as JSON: {json_filename}")
except Exception as e:
    print(f"❌ Error saving JSON file: {e}")

print(f"\n📊 Backup Summary:")
print(f"   - Total segments: {len(speaker_segments)}")
print(f"   - Pickle file: {backup_filename}")
print(f"   - JSON file: {json_filename}")
print(f"   - Audio source: {output_wav_file if 'output_wav_file' in globals() else 'unknown'}")

# Function to load speaker segments from backup
def load_speaker_segments_backup(filename):
    """Load speaker segments from a backup file"""
    try:
        if filename.endswith('.pkl'):
            with open(filename, 'rb') as f:
                segments = pickle.load(f)
            print(f"✓ Loaded {len(segments)} segments from pickle: {filename}")
        elif filename.endswith('.json'):
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            segments = data.get('segments', [])
            metadata = data.get('metadata', {})
            print(f"✓ Loaded {len(segments)} segments from JSON: {filename}")
            print(f"   Created: {metadata.get('created', 'unknown')}")
            print(f"   Source audio: {metadata.get('audio_file', 'unknown')}")
        else:
            raise ValueError("File must be .pkl or .json")
        
        return segments
    except Exception as e:
        print(f"❌ Error loading backup: {e}")
        return None

print(f"\n💡 To restore from backup later, use:")
print(f"   speaker_segments = load_speaker_segments_backup('{backup_filename}')")
print(f"   # or")
print(f"   speaker_segments = load_speaker_segments_backup('{json_filename}')")

## Diarization Performance Tips

### Current Performance Expectations:
- **With GPU**: ~2-4x real-time (10 min audio → 20-40 min processing)
- **With CPU only**: ~5-15x real-time (10 min audio → 50-150 min processing)

### Speed Optimization Options:

1. **Use GPU**: Install CUDA and PyTorch with GPU support
   ```bash
   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
   ```

2. **Pre-process Audio**: 
   - Convert to mono (reduces processing by ~50%)
   - Downsample to 16kHz (reduces processing time)
   - Remove silence/very quiet sections

3. **Alternative Models**:
   - Use faster but less accurate models for quick tests
   - Consider cloud APIs (Azure/Google) for very long audio

4. **Split Long Audio**: 
   - Process in chunks (e.g., 10-15 minute segments)
   - Parallel processing of chunks

In [ ]:
# Optional: Helper functions for analysis
def analyze_speaker_stats(speaker_segments):
    """Analyze speaking time per speaker"""
    speaker_time = {}
    total_time = 0
    
    for segment in speaker_segments:
        duration = segment['end'] - segment['start']
        speaker = segment['speaker']
        
        if speaker not in speaker_time:
            speaker_time[speaker] = 0
        speaker_time[speaker] += duration
        total_time += duration
    
    print("\n=== Speaker Analysis ===")
    for speaker, time in sorted(speaker_time.items()):
        percentage = (time / total_time) * 100 if total_time > 0 else 0
        print(f"{speaker}: {time:.1f}s ({percentage:.1f}%)")
    
    return speaker_time

def filter_short_segments(speaker_segments, min_duration=2.0):
    """Filter out very short segments that might be noise"""
    filtered = []
    for segment in speaker_segments:
        duration = segment['end'] - segment['start']
        if duration >= min_duration:
            filtered.append(segment)
        else:
            print(f"Skipping short segment: {segment['speaker']} ({duration:.1f}s)")
    
    print(f"Filtered {len(speaker_segments)} -> {len(filtered)} segments")
    return filtered

In [ ]:
# Performance optimizations - run this before transcription
def optimize_segments(speaker_segments, min_duration=3.0, max_segments=None):
    """
    Optimize speaker segments for faster processing:
    - Filter out very short segments (likely noise/filler words)
    - Optionally limit total segments for testing
    - Merge very close segments from the same speaker
    """
    print(f"Original segments: {len(speaker_segments)}")
    
    # Filter by minimum duration
    filtered = [s for s in speaker_segments if (s['end'] - s['start']) >= min_duration]
    print(f"After duration filter (≥{min_duration}s): {len(filtered)}")
    
    # Limit segments for testing (optional)
    if max_segments and len(filtered) > max_segments:
        filtered = filtered[:max_segments]
        print(f"Limited to first {max_segments} segments for testing")
    
    # Merge close segments from same speaker (within 1 second)
    merged = []
    for segment in filtered:
        if (merged and 
            merged[-1]['speaker'] == segment['speaker'] and 
            segment['start'] - merged[-1]['end'] <= 1.0):
            # Merge with previous segment
            merged[-1]['end'] = segment['end']
        else:
            merged.append(segment.copy())
    
    print(f"After merging close segments: {len(merged)}")
    return merged

# Apply optimizations - adjust parameters as needed
speaker_segments = optimize_segments(
    speaker_segments, 
    min_duration=3.0,    # Skip segments shorter than 3 seconds
    # max_segments=20      # For testing - remove this line for full processing
)

# Show speaker statistics after optimization
analyze_speaker_stats(speaker_segments)

## Transcription Configuration

Choose your transcription mode:

### Option 1: Speaker Diarization Mode (Recommended)
- **Pros**: Identifies who is speaking, more accurate speaker boundaries
- **Cons**: Requires diarization step (can be slow)
- **Best for**: Meetings with multiple speakers where you need to know who said what

### Option 2: Time-based Chunking Mode
- **Pros**: Fast, no diarization required, works with any audio
- **Cons**: No speaker identification, fixed time segments
- **Best for**: Single speaker recordings, quick transcription, or when diarization fails

Configure the transcription behavior in the next cell by setting `USE_DIARIZATION = True/False`.

In [ ]:
import os
from pydub import AudioSegment
from pydub.utils import make_chunks
import speech_recognition as sr
import tempfile
import time

# Configuration
USE_DIARIZATION = True  # Set to False to skip diarization and use time-based chunks
CHUNK_LENGTH_MS = 60000  # 1 minute chunks (only used when diarization is disabled)

# Path to your audio file
AUDIO_FILE = output_wav_file

# Load the full audio file
full_audio = AudioSegment.from_file(AUDIO_FILE)

recognizer = sr.Recognizer()
# Optimize recognizer settings for faster processing
recognizer.energy_threshold = 300
recognizer.dynamic_energy_threshold = True
recognizer.pause_threshold = 0.8

output_file = f"transcription_{timestamp}.txt"
transcribed_count = 0

# Determine segments to process
if USE_DIARIZATION and "speaker_segments" in globals() and speaker_segments:
    print("Using speaker diarization segments...")
    segments_to_process = speaker_segments
    segment_type = "speaker"
else:
    print(
        "Diarization disabled or no speaker segments found. Using time-based chunks..."
    )
    USE_DIARIZATION = False

    # Create time-based chunks
    chunks = make_chunks(full_audio, CHUNK_LENGTH_MS)
    segments_to_process = []

    for i, chunk in enumerate(chunks):
        start_time = i * (CHUNK_LENGTH_MS / 1000)
        end_time = min((i + 1) * (CHUNK_LENGTH_MS / 1000), len(full_audio) / 1000)

        segments_to_process.append(
            {
                "start": start_time,
                "end": end_time,
                "speaker": f"CHUNK_{i+1}",
                "audio_chunk": chunk,  # Store the chunk directly for efficiency
            }
        )

    segment_type = "time-chunk"

total_segments = len(segments_to_process)
print(f"Processing {total_segments} {segment_type} segments...")

start_time = time.time()

# Create a single temporary directory for all files
temp_dir = tempfile.mkdtemp()

try:
    with open(output_file, "w", encoding="utf-8") as f:
        for i, segment in enumerate(segments_to_process):
            segment_start_time = time.time()

            try:
                if USE_DIARIZATION:
                    # Extract audio segment for speaker diarization
                    start_ms = int(segment["start"] * 1000)
                    end_ms = int(segment["end"] * 1000)
                    audio_chunk = full_audio[start_ms:end_ms]
                    speaker_label = segment["speaker"]
                else:
                    # Use pre-created chunk for time-based segmentation
                    audio_chunk = segment["audio_chunk"]
                    speaker_label = segment["speaker"]

                # Skip very quiet segments
                if audio_chunk.max_possible_amplitude > 0:
                    if audio_chunk.rms < audio_chunk.max_possible_amplitude * 0.01:
                        print(f"Skipping silent segment {i+1}")
                        continue

                # Create temp file in the temp directory
                temp_filename = os.path.join(temp_dir, f"segment_{i}.wav")
                audio_chunk.export(temp_filename, format="wav")

                # Transcribe this segment
                with sr.AudioFile(temp_filename) as source:
                    # Adjust for ambient noise (only once)
                    if i == 0:
                        recognizer.adjust_for_ambient_noise(source, duration=0.5)

                    audio_data = recognizer.record(source)
                    retries = 2

                    for attempt in range(retries):
                        try:
                            text = recognizer.recognize_google(
                                audio_data, language="en-US"
                            )

                            # Format time stamps
                            start_min = int(segment["start"] // 60)
                            start_sec = int(segment["start"] % 60)
                            end_min = int(segment["end"] // 60)
                            end_sec = int(segment["end"] % 60)

                            # Write timestamp, speaker/chunk, and transcription
                            if USE_DIARIZATION:
                                f.write(
                                    f"[{start_min:02d}:{start_sec:02d} - {end_min:02d}:{end_sec:02d}] "
                                    f"{speaker_label}: {text}\n"
                                )
                            else:
                                f.write(
                                    f"[{start_min:02d}:{start_sec:02d} - {end_min:02d}:{end_sec:02d}] {text}\n"
                                )

                            segment_time = time.time() - segment_start_time
                            print(
                                f"✓ Segment {i+1}/{total_segments} ({speaker_label}) "
                                f"[{segment_time:.1f}s]: {text[:50]}..."
                            )
                            transcribed_count += 1
                            break

                        except sr.UnknownValueError:
                            if attempt == retries - 1:
                                # Write a placeholder for failed transcription
                                start_min = int(segment["start"] // 60)
                                start_sec = int(segment["start"] % 60)
                                end_min = int(segment["end"] // 60)
                                end_sec = int(segment["end"] % 60)

                                if USE_DIARIZATION:
                                    f.write(
                                        f"[{start_min:02d}:{start_sec:02d} - {end_min:02d}:{end_sec:02d}] "
                                        f"{speaker_label}: [INAUDIBLE]\n"
                                    )
                                else:
                                    f.write(
                                        f"[{start_min:02d}:{start_sec:02d} - {end_min:02d}:{end_sec:02d}] [INAUDIBLE]\n"
                                    )
                                print(f"✗ Segment {i+1} ({speaker_label}) - inaudible")

                        except sr.RequestError as e:
                            print(f"✗ API Error for segment {i+1}: {e}")
                            break

                # Clean up individual temp file
                if os.path.exists(temp_filename):
                    os.remove(temp_filename)

            except Exception as e:
                print(f"✗ Error processing segment {i+1}: {e}")
                continue

finally:
    # Clean up temp directory
    import shutil

    shutil.rmtree(temp_dir, ignore_errors=True)

total_time = time.time() - start_time
print(f"\n=== Transcription Complete ===")
print(f"Mode: {'Speaker Diarization' if USE_DIARIZATION else 'Time-based Chunking'}")
print(f"Total time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"Output saved to: {output_file}")

percentage = (transcribed_count / total_segments) * 100 if total_segments > 0 else 0
print(
    f"Successfully transcribed: {transcribed_count}/{total_segments} segments ({percentage:.1f}%)"
)
print(f"Average time per segment: {total_time/total_segments:.1f}s")

## Meeting Summary Generation

This section uses Ollama to generate intelligent summaries from the transcription.

### Summary Options:

1. **Basic Summary**: Questions, suggestions, and key discussions
2. **Detailed Analysis**: Speaker contributions, action items, decisions made
3. **Custom Prompt**: Use your own prompt template

### Requirements:
- Ollama must be installed and running locally
- Choose an appropriate model (gemma2, llama3, mistral, etc.)
- Adjust prompt based on your meeting type

In [ ]:
import ollama
import time
import os

# Configuration
OLLAMA_MODEL = "gpt-oss:20b"  # Options: gemma3, llama3, mistral, qwen2, etc.
SUMMARY_TYPE = "custom"  # Options: "basic", "detailed", "custom"
OUTPUT_FORMAT = "markdown"  # Options: "markdown", "text"

def get_summary_prompt(transcript, summary_type="basic"):
    """Generate appropriate prompt based on summary type"""
    
    base_context = (
        "You are an expert meeting analyst. The following is a meeting transcription. "
        "The text may contain some errors from speech recognition processing. "
        "Please analyze and improve the content where needed.\n\n"
    )
    
    if summary_type == "basic":
        instruction = (
            "Provide a concise summary focusing on:\n"
            "- Key questions asked\n"
            "- Suggestions and recommendations made\n"
            "- Main ideas and topics discussed\n"
            "- Important decisions or conclusions\n\n"
        )
    
    elif summary_type == "detailed":
        instruction = (
            "Provide a comprehensive analysis including:\n"
            "- Executive Summary (2-3 sentences)\n"
            "- Key Participants and their main contributions\n"
            "- Questions Asked (with context)\n"
            "- Suggestions and Recommendations\n"
            "- Decisions Made\n"
            "- Action Items (if any)\n"
            "- Next Steps or Follow-ups mentioned\n"
            "- Key Topics/Themes discussed\n\n"
        )
    
    else:  # custom
        instruction = (
            "Analyze this meeting transcript and provide a textual summary of what was dicussed, using two or three paragraphs. Write the text saying \"we discussed\" instead of \"the participants discussed\" or similar phrases."
        )
    
    return base_context + instruction + f"Transcript:\n\n{transcript}"

def check_ollama_connection():
    """Check if Ollama is running and accessible"""
    try:
        models = ollama.list()
        return True, models
    except Exception as e:
        return False, str(e)

def generate_summary_with_retry(model, prompt, max_retries=3):
    """Generate summary with retry logic"""
    for attempt in range(max_retries):
        try:
            print(f"Generating summary (attempt {attempt + 1}/{max_retries})...")
            start_time = time.time()
            
            response = ollama.generate(
                model=model, 
                prompt=prompt,
                options={
                    "temperature": 0.3,  # More focused responses
                    "top_p": 0.9,
                    # "num_ctx": 4096  # Context window
                }
            )
            
            generation_time = time.time() - start_time
            print(f"Summary generated in {generation_time:.1f} seconds")
            return response["response"]
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                raise e
            time.sleep(2)  # Wait before retry

# Main summarization process
print("=== Meeting Summary Generation ===")

# Check transcription file
if not os.path.exists(f"transcription_{timestamp}.txt"):
    print("❌ Error: transcription.txt not found. Please run transcription first.")
else:
    # Check Ollama connection
    print("Checking Ollama connection...")
    is_connected, result = check_ollama_connection()
    
    if not is_connected:
        print(f"❌ Error: Cannot connect to Ollama. {result}")
        print("Make sure Ollama is installed and running (try: ollama serve)")
    else:
        # Handle different response formats from ollama.list()
        if hasattr(result, 'models') and result.models:
            available_models = [model.model if hasattr(model, 'model') else str(model) for model in result.models]
        elif isinstance(result, dict) and 'models' in result:
            available_models = [model.get('name', model.get('model', str(model))) for model in result['models']]
        else:
            # Fallback - try to extract model names from the result
            available_models = [str(model) for model in (result if isinstance(result, list) else [result])]
        
        print(f"✓ Ollama connected. Available models: {', '.join(available_models)}")
        
        if OLLAMA_MODEL not in available_models:
            if available_models:
                print(f"⚠️  Model '{OLLAMA_MODEL}' not found. Using first available model: {available_models[0]}")
                OLLAMA_MODEL = available_models[0]
            else:
                print(f"⚠️  No models found. Please install a model first (e.g., ollama pull {OLLAMA_MODEL})")
                OLLAMA_MODEL = OLLAMA_MODEL  # Keep original for error message
        
        # Read transcription
        with open("transcription.txt", "r", encoding="utf-8") as f:
            transcript = f.read().strip()
        
        if not transcript:
            print("❌ Error: Transcription file is empty.")
        else:
            transcript_length = len(transcript)
            print(f"📄 Transcript loaded: {transcript_length} characters")
            
            # Generate summary prompt
            prompt = get_summary_prompt(transcript, SUMMARY_TYPE)
            print(f"🎯 Using {SUMMARY_TYPE} summary with {OLLAMA_MODEL} model")
            
            try:
                # Generate summary
                summary = generate_summary_with_retry(OLLAMA_MODEL, prompt)
                
                # Display summary
                print("\n" + "="*60)
                print("📋 MEETING SUMMARY")
                print("="*60)
                print(summary)
                print("="*60)
                
                # Save summary to file
                output_extension = "md" if OUTPUT_FORMAT == "markdown" else "txt"
                summary_file = f"summary_{timestamp}.{output_extension}"
                
                with open(summary_file, "w", encoding="utf-8") as f:
                    if OUTPUT_FORMAT == "markdown":
                        f.write(f"# Meeting Summary\n\n")
                        f.write(f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                        f.write(f"**Model:** {OLLAMA_MODEL}\n")
                        f.write(f"**Summary Type:** {SUMMARY_TYPE}\n\n")
                        f.write("---\n\n")
                    f.write(summary)
                
                print(f"💾 Summary saved to: {summary_file}")
                
                # Statistics
                summary_length = len(summary)
                compression_ratio = transcript_length / summary_length if summary_length > 0 else 0
                print(f"📊 Compression ratio: {compression_ratio:.1f}:1 (Original: {transcript_length} → Summary: {summary_length} characters)")
                
            except Exception as e:
                print(f"❌ Error generating summary: {e}")
                print("Try reducing the transcript length or using a different model.")

In [ ]:
# Optional: Additional summary utilities and analysis

def extract_action_items(summary_text):
    """Extract potential action items from the summary"""
    action_keywords = [
        "should", "will", "need to", "must", "action item", 
        "follow up", "next step", "todo", "assign", "deadline"
    ]
    
    lines = summary_text.split('\n')
    potential_actions = []
    
    for line in lines:
        line_lower = line.lower()
        if any(keyword in line_lower for keyword in action_keywords):
            potential_actions.append(line.strip())
    
    return potential_actions

def generate_meeting_report():
    """Generate a comprehensive meeting report combining transcript and summary"""
    if not os.path.exists(f"transcription_{timestamp}.txt") or not os.path.exists(f"summary_{timestamp}.md"):
        print("Missing required files for report generation")
        return
    
    with open(f"transcription_{timestamp}.txt", "r", encoding="utf-8") as f:
        transcript = f.read()
    
    with open(f"summary_{timestamp}.md", "r", encoding="utf-8") as f:
        summary = f.read()
    
    # Extract action items
    action_items = extract_action_items(summary)
    
    # Generate comprehensive report
    report = f"""# Meeting Report
**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}

## Summary
{summary}

## Action Items Identified
"""
    
    if action_items:
        for i, item in enumerate(action_items, 1):
            report += f"{i}. {item}\n"
    else:
        report += "No specific action items identified in this meeting.\n"
    
    report += f"""

## Full Transcript
```
{transcript}
```

---
*Report generated automatically by Meeting Processor*
"""
    
    with open(f"meeting_report_{timestamp}.md", "w", encoding="utf-8") as f:
        f.write(report)
    
    print(f"📋 Comprehensive meeting report saved to: meeting_report_{timestamp}.md")
    return len(action_items)

# Optional: Generate additional outputs
print("\n=== Additional Analysis ===")

if os.path.exists(f"summary_{timestamp}.md"):
    with open(f"summary_{timestamp}.md", "r", encoding="utf-8") as f:
        summary_content = f.read()
    
    # Extract action items
    actions = extract_action_items(summary_content)
    if actions:
        print(f"🎯 Found {len(actions)} potential action items:")
        for i, action in enumerate(actions[:5], 1):  # Show first 5
            print(f"   {i}. {action[:80]}{'...' if len(action) > 80 else ''}")
        if len(actions) > 5:
            print(f"   ... and {len(actions) - 5} more")
    
    # Generate comprehensive report
    action_count = generate_meeting_report()
    print(f"📊 Total action items identified: {action_count}")

else:
    print("⚠️  Summary not found. Run the previous cell first.")